# Ethiopia EDA

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)


## Load

In [ ]:
df = pd.read_csv("../data/ethiopia.csv")
df["Country"] = "Ethiopia"

df["Date"] = pd.to_datetime(df["YEAR"] * 1000 + df["DOY"], format="%Y%j")
df["Month"] = df["Date"].dt.month
df.head()

## Clean

In [ ]:
df.replace(-999, np.nan, inplace=True)
df = df.drop_duplicates()
print(df.describe())

missing_pct = df.isna().mean() * 100
print(missing_pct[missing_pct > 0])


## Outliers

In [ ]:
weather_cols = ['T2M', 'T2M_MAX', 'T2M_MIN', 'PRECTOTCORR', 'RH2M', 'WS2M', 'WS2M_MAX']
z = np.abs(stats.zscore(df[weather_cols].dropna(), nan_policy='omit'))
z_df = pd.DataFrame(z, index=df[weather_cols].dropna().index, columns=weather_cols)
print((z_df > 3).sum())

df = df[df.isna().mean(axis=1) <= 0.3].copy()
df[weather_cols] = df[weather_cols].ffill().bfill()

if 'T2M_RANGE' not in df.columns:
        df['T2M_RANGE'] = df['T2M_MAX'] - df['T2M_MIN']
        
df.to_csv('../data/ethiopia_clean.csv', index=False)
print('saved')


## Time series

In [ ]:
monthly_temp = df.set_index('Date').resample('M')['T2M'].mean().dropna()
warmest = monthly_temp.idxmax()
coolest = monthly_temp.idxmin()

plt.figure()
plt.plot(monthly_temp.index, monthly_temp.values, marker='o')
plt.title('Monthly avg T2M')
plt.xlabel('Date')
plt.ylabel('T2M')
plt.scatter([warmest], [monthly_temp.loc[warmest]], color='red')
plt.scatter([coolest], [monthly_temp.loc[coolest]], color='blue')
plt.show()


In [ ]:

monthly_precip = df.groupby('Month')['PRECTOTCORR'].sum()
plt.figure()
plt.bar(monthly_precip.index, monthly_precip.values, color='skyblue')
plt.title('Monthly total PRECTOTCORR')
plt.xlabel('Month')
plt.ylabel('PRECTOTCORR')
plt.show()



## Correlation

In [ ]:
num = df.select_dtypes(include=[np.number])
corr = num.corr()
plt.figure(figsize=(10, 8))
sns.heatmap(corr, cmap='coolwarm', center=0, annot=False)
plt.title('Correlation heatmap')
plt.show()

In [ ]:

print(corr.unstack().sort_values(ascending=False)[len(corr):len(corr)+3])
plt.figure()
sns.scatterplot(data=df, x='T2M', y='RH2M', alpha=0.5)
plt.title('T2M vs RH2M')
plt.show()

In [ ]:

plt.figure()
sns.scatterplot(data=df, x='T2M_RANGE', y='WS2M', alpha=0.5)
plt.title('T2M_RANGE vs WS2M')
plt.show()


## Distribution

In [ ]:
plt.figure()
sns.histplot(df['PRECTOTCORR'].fillna(0) + 1e-6, bins=50, log_scale=True, color='dodgerblue')
plt.title('PRECTOTCORR distribution')
plt.show()


In [ ]:
plt.figure()
df2 = df[df['PRECTOTCORR'] > 0]
sns.scatterplot(data=df2, x='T2M', y='RH2M', size='PRECTOTCORR', sizes=(20, 300), alpha=0.5, hue='PRECTOTCORR', palette='viridis', legend=False)
plt.title('T2M vs RH2M bubble')
plt.show()